# Merge LINCS metadata, ChEMBL target mask, and ctl_vehicle baseline expression 

이 노트북은 기존 `metadata_X`와 ChEMBL target mapping 결과를 병합하고, `siginfo_beta.txt`에서 `ctl_vehicle` sample의 `sig_id`를 찾아 `level5_ctl.gctx`에서 baseline/diseased-like expression을 읽어옵니다.

이번 수정에서는 **모든 gene feature를 LINCS 978 landmark gene 기준으로 통일**합니다.

수정한 핵심 오류는 다음입니다.

1. `siginfo_beta.txt`에서 `sample_id`를 만들지 않고 **반드시 `sig_id`를 사용**합니다.
2. `ctl_gct`는 `GCToo` 객체이므로 `.head()`가 아니라 `ctl_gct.data_df`를 사용합니다.
3. `.gctx`에 실제 존재하는 `sig_id`만 필터링한 뒤 `parse(..., cid=...)`를 실행합니다.
4. 기존 ChEMBL preprocessing에서 전체 gene 기준으로 만들어졌을 수 있는 `target_positions_lincs`를 그대로 쓰지 않고, `geneinfo_beta.txt`의 `feature_space == "landmark"` 978개 gene 기준으로 target 위치를 다시 계산합니다.


In [1]:
import os
import re
import pandas as pd
import numpy as np
from cmapPy.pandasGEXpress.parse import parse

## 1. 경로 설정

`META_BASE`는 기존 LINCS metadata, `META_CHEMBL`은 ChEMBL target mapping 결과입니다.  
`CTL_GCTX`는 `ctl_vehicle` signature가 들어 있는 GCTX 파일이어야 합니다.

In [ ]:
META_BASE = "./lincs/metadata_X.csv"
META_CHEMBL = "./chembl/metadata_X_with_chembl_targets.csv"

CTL_GCTX = "./data/level5_ctl.gctx"
SIGINFO = "./data/siginfo_beta.txt"
GENEINFO = "./data/geneinfo_beta.txt"

OUT_DIR = "./processed/result"
os.makedirs(OUT_DIR, exist_ok=True)

## 2. LINCS metadata와 ChEMBL target metadata 로드

CSV 저장 과정에서 index가 `Unnamed: 0`으로 들어간 경우 이를 `sample_id`로 바꿉니다.  
단, `siginfo_beta.txt`에는 이 처리를 적용하지 않습니다. `siginfo_beta.txt`에서는 `sig_id`를 사용해야 합니다.

In [3]:
def ensure_sample_id(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if "sample_id" in df.columns:
        return df
    if "Unnamed: 0" in df.columns:
        return df.rename(columns={"Unnamed: 0": "sample_id"})
    raise ValueError("sample_id 컬럼이 없고 Unnamed: 0 컬럼도 없습니다. 입력 파일을 확인하세요.")

meta_base = pd.read_csv(META_BASE, low_memory=False)
meta_chembl = pd.read_csv(META_CHEMBL, low_memory=False)

meta_base = ensure_sample_id(meta_base)
meta_chembl = ensure_sample_id(meta_chembl)

meta_base["sample_id"] = meta_base["sample_id"].astype(str)
meta_chembl["sample_id"] = meta_chembl["sample_id"].astype(str)

print("meta_base:", meta_base.shape)
print("meta_chembl:", meta_chembl.shape)
display(meta_base.head())
display(meta_chembl.head())

meta_base: (35149, 39)
meta_chembl: (10670, 48)


,sample_id,bead_batch,nearest_dose,pert_dose,pert_dose_unit,pert_idose,pert_itime,pert_time,pert_time_unit,cell_mfc_name,...,det_plates,distil_ids,build_name,project_code,cmap_name,is_exemplar_sig,is_ncs_sig,is_null_sig,pert_time_num,pert_dose_num
0,ABY001_A549_XH:BRD-A61304759:0.625:24,b15,0.66,0.625,uM,0.66 uM,24 h,24.0,h,A549,...,ABY001_A549_XH_X1_B15,ABY001_A549_XH_X1_B15:H15|ABY001_A549_XH_X1_B1...,NaN,ABY,tanespimycin,0,1.0,0.0,24.0,0.625
1,ABY001_A549_XH:BRD-A61304759:10:24,b15,10.00,10.000,uM,10 uM,24 h,24.0,h,A549,...,ABY001_A549_XH_X1_B15,ABY001_A549_XH_X1_B15:H13|ABY001_A549_XH_X1_B1...,NaN,ABY,tanespimycin,0,1.0,0.0,24.0,10.000
2,ABY001_A549_XH:BRD-A61304759:2.5:24,b15,2.50,2.500,uM,2.5 uM,24 h,24.0,h,A549,...,ABY001_A549_XH_X1_B15,ABY001_A549_XH_X1_B15:H14|ABY001_A549_XH_X1_B1...,NaN,ABY,tanespimycin,0,1.0,0.0,24.0,2.500
3,ABY001_A549_XH:BRD-A90490067:0.625:24,b15,0.66,0.625,uM,0.66 uM,24 h,24.0,h,A549,...,ABY001_A549_XH_X1_B15,ABY001_A549_XH_X1_B15:F15|ABY001_A549_XH_X1_B1...,NaN,ABY,fulvestrant,0,1.0,0.0,24.0,0.625
4,ABY001_A549_XH:BRD-A90490067:10:24,b15,10.00,10.000,uM,10 uM,24 h,24.0,h,A549,...,ABY001_A549_XH_X1_B15,ABY001_A549_XH_X1_B15:D15|ABY001_A549_XH_X1_B1...,NaN,ABY,fulvestrant,0,1.0,0.0,24.0,10.000


,sample_id,bead_batch,nearest_dose,pert_dose,pert_dose_unit,pert_idose,pert_itime,pert_time,pert_time_unit,cell_mfc_name,...,pert_dose_num,cmap_name_norm,molecule_chembl_id,drug_name,drug_name_norm,chembl_target_genes,n_chembl_targets,target_genes_lincs,target_positions_lincs,n_lincs_targets
0,ABY001_A549_XH:BRD-A90490067:0.625:24,b15,0.66,0.625,uM,0.66 uM,24 h,24.0,h,A549,...,0.625,fulvestrant,CHEMBL1358,FULVESTRANT,fulvestrant,ESR1,1,ESR1,4182,1
1,ABY001_A549_XH:BRD-A90490067:10:24,b15,10.00,10.000,uM,10 uM,24 h,24.0,h,A549,...,10.000,fulvestrant,CHEMBL1358,FULVESTRANT,fulvestrant,ESR1,1,ESR1,4182,1
2,ABY001_A549_XH:BRD-A90490067:2.5:24,b15,2.50,2.500,uM,2.5 uM,24 h,24.0,h,A549,...,2.500,fulvestrant,CHEMBL1358,FULVESTRANT,fulvestrant,ESR1,1,ESR1,4182,1
3,ABY001_A549_XH:BRD-K70511574:0.625:24,b15,0.66,0.625,uM,0.66 uM,24 h,24.0,h,A549,...,0.625,hmn214,CHEMBL3991927,HMN-214,hmn214,PLK1,1,PLK1,2467,1
4,ABY001_A549_XH:BRD-K70511574:10:24,b15,10.00,10.000,uM,10 uM,24 h,24.0,h,A549,...,10.000,hmn214,CHEMBL3991927,HMN-214,hmn214,PLK1,1,PLK1,2467,1


## 3. ChEMBL target 정보 병합

`target_positions_lincs`는 각 drug target gene이 LINCS gene vector의 어느 위치에 매핑되는지 나타냅니다.

In [4]:
target_cols = [
    "sample_id",
    "cmap_name",
    "molecule_chembl_id",
    "chembl_target_genes",
    "n_chembl_targets",
    "target_genes_lincs",
    "target_positions_lincs",
    "n_lincs_targets",
]

missing_cols = [c for c in target_cols if c not in meta_chembl.columns]
if missing_cols:
    raise ValueError(f"meta_chembl에 필요한 컬럼이 없습니다: {missing_cols}")

meta_target = meta_chembl[target_cols].copy()

# sample_id가 중복된 경우 첫 번째 항목만 사용
meta_target = meta_target.drop_duplicates(subset=["sample_id"], keep="first")

meta = meta_base.merge(
    meta_target,
    on="sample_id",
    how="inner",
    validate="one_to_one"
)

print("merged meta:", meta.shape)
display(meta.head())

merged meta: (10670, 46)


,sample_id,bead_batch,nearest_dose,pert_dose,pert_dose_unit,pert_idose,pert_itime,pert_time,pert_time_unit,cell_mfc_name,...,is_null_sig,pert_time_num,pert_dose_num,cmap_name_y,molecule_chembl_id,chembl_target_genes,n_chembl_targets,target_genes_lincs,target_positions_lincs,n_lincs_targets
0,ABY001_A549_XH:BRD-A90490067:0.625:24,b15,0.66,0.625,uM,0.66 uM,24 h,24.0,h,A549,...,0.0,24.0,0.625,fulvestrant,CHEMBL1358,ESR1,1,ESR1,4182,1
1,ABY001_A549_XH:BRD-A90490067:10:24,b15,10.00,10.000,uM,10 uM,24 h,24.0,h,A549,...,0.0,24.0,10.000,fulvestrant,CHEMBL1358,ESR1,1,ESR1,4182,1
2,ABY001_A549_XH:BRD-A90490067:2.5:24,b15,2.50,2.500,uM,2.5 uM,24 h,24.0,h,A549,...,0.0,24.0,2.500,fulvestrant,CHEMBL1358,ESR1,1,ESR1,4182,1
3,ABY001_A549_XH:BRD-K70511574:0.625:24,b15,0.66,0.625,uM,0.66 uM,24 h,24.0,h,A549,...,0.0,24.0,0.625,HMN-214,CHEMBL3991927,PLK1,1,PLK1,2467,1
4,ABY001_A549_XH:BRD-K70511574:10:24,b15,10.00,10.000,uM,10 uM,24 h,24.0,h,A549,...,0.0,24.0,10.000,HMN-214,CHEMBL3991927,PLK1,1,PLK1,2467,1


## 4. ChEMBL target gene list 파싱

기존 ChEMBL preprocessing 파일의 `target_positions_lincs`는 전체 gene 기준으로 계산되었을 가능성이 있습니다.
따라서 여기서는 position을 바로 쓰지 않고, 먼저 `chembl_target_genes` 문자열만 gene list로 파싱합니다.

이후 `geneinfo_beta.txt`에서 978개 landmark gene을 로드한 뒤, landmark gene 기준 position을 새로 계산합니다.


In [8]:
def parse_gene_list(x):
    """Parse target gene strings saved with delimiters such as ||, comma, or semicolon."""
    if pd.isna(x):
        return []
    s = str(x).strip()
    if s == "" or s.lower() == "nan":
        return []
    parts = re.split(r"\|\||,|;", s)
    out = []
    for p in parts:
        gene = p.strip().strip("[]'\" ")
        if gene:
            out.append(gene)
    return out

meta["chembl_target_gene_list"] = meta["chembl_target_genes"].apply(parse_gene_list)

print("meta after merge:", meta.shape)
print("samples with raw ChEMBL target genes:", (meta["chembl_target_gene_list"].apply(len) > 0).sum())
display(meta[["sample_id", "cell_iname", "cmap_name_x", "chembl_target_genes", "chembl_target_gene_list"]].head())


meta after merge: (10670, 47)
samples with raw ChEMBL target genes: 10670


,sample_id,cell_iname,cmap_name_x,chembl_target_genes,chembl_target_gene_list
0,ABY001_A549_XH:BRD-A90490067:0.625:24,A549,fulvestrant,ESR1,[ESR1]
1,ABY001_A549_XH:BRD-A90490067:10:24,A549,fulvestrant,ESR1,[ESR1]
2,ABY001_A549_XH:BRD-A90490067:2.5:24,A549,fulvestrant,ESR1,[ESR1]
3,ABY001_A549_XH:BRD-K70511574:0.625:24,A549,HMN-214,PLK1,[PLK1]
4,ABY001_A549_XH:BRD-K70511574:10:24,A549,HMN-214,PLK1,[PLK1]


## 5. siginfo와 geneinfo 로드

`siginfo_beta.txt`는 `sig_id`가 핵심 key입니다.  
첫 번째 컬럼을 임의로 `sample_id`로 rename하면 `bead_batch` 값인 `b10`, `b15` 등이 sample id처럼 사용되어 `.gctx` parsing 오류가 납니다.

In [9]:
ctl_meta = pd.read_csv(SIGINFO, sep="	", low_memory=False)
gene_info = pd.read_csv(GENEINFO, sep="	", low_memory=False)

if "sig_id" not in ctl_meta.columns:
    raise ValueError("siginfo_beta.txt에 sig_id 컬럼이 없습니다.")

required_sig_cols = ["sig_id", "pert_type", "cell_iname", "pert_time"]
missing = [c for c in required_sig_cols if c not in ctl_meta.columns]
if missing:
    raise ValueError(f"siginfo_beta.txt에 필요한 컬럼이 없습니다: {missing}")

landmark_gene_info = gene_info[gene_info["feature_space"] == "landmark"].copy()
landmark_gene_info = landmark_gene_info[["gene_id", "gene_symbol"]].dropna().copy()
landmark_gene_info["gene_id"] = landmark_gene_info["gene_id"].astype(str)
landmark_gene_info["gene_symbol"] = landmark_gene_info["gene_symbol"].astype(str)

landmark_gene_ids = landmark_gene_info["gene_id"].tolist()
landmark_gene_symbols = landmark_gene_info["gene_symbol"].tolist()
n_genes = len(landmark_gene_ids)

print("ctl_meta:", ctl_meta.shape)
print("landmark genes:", n_genes)
display(ctl_meta[["sig_id", "pert_type", "cell_iname", "pert_time"]].head())
display(landmark_gene_info.head())

ctl_meta: (1201944, 37)
landmark genes: 978


,sig_id,pert_type,cell_iname,pert_time
0,MET001_N8_XH:BRD-U44432129:100:336,trt_cp,NAMEC8,336.0
1,ABY001_A549_XH:BRD-K81418486:10:3,trt_cp,A549,3.0
2,ABY001_HT29_XH:BRD-K70511574:2.5:24,trt_cp,HT29,24.0
3,LTC002_HME1_3H:BRD-K81418486:10,trt_cp,HME1,3.0
4,ABY001_H1975_XH:BRD-A61304759:10:3,trt_cp,H1975,3.0


,gene_id,gene_symbol
2154,16,AARS
2155,23,ABCF1
2156,25,ABL1
2157,30,ACAA1
2158,39,ACAT2


## 5-1. ChEMBL target을 978 landmark gene 기준으로 다시 매핑

`chembl_preprocessing.ipynb`에서 만들어진 `target_positions_lincs`는 전체 `geneinfo_beta.txt` 기준일 수 있습니다.  
따라서 이 노트북에서는 `chembl_target_genes`를 다시 읽고, `feature_space == "landmark"`인 978개 gene symbol에만 매핑합니다.


In [11]:
# Landmark gene symbol -> position mapping
landmark_symbol_to_pos = {
    symbol.upper(): idx
    for idx, symbol in enumerate(landmark_gene_symbols)
}

landmark_symbol_upper_to_original = {
    symbol.upper(): symbol
    for symbol in landmark_gene_symbols
}

def map_targets_to_landmark(target_genes):
    mapped_genes = []
    mapped_positions = []
    seen = set()

    for gene in target_genes:
        g = str(gene).strip().upper()
        if g in landmark_symbol_to_pos and g not in seen:
            seen.add(g)
            mapped_genes.append(landmark_symbol_upper_to_original[g])
            mapped_positions.append(landmark_symbol_to_pos[g])

    return mapped_genes, mapped_positions

mapped = meta["chembl_target_gene_list"].apply(map_targets_to_landmark)

meta["target_genes_lincs_landmark"] = mapped.apply(lambda x: x[0])
meta["target_positions_lincs_landmark"] = mapped.apply(lambda x: x[1])
meta["n_lincs_targets_landmark"] = meta["target_positions_lincs_landmark"].apply(len)

before = meta.shape[0]
meta = meta[meta["n_lincs_targets_landmark"] > 0].copy()
after = meta.shape[0]

print("samples before landmark target filtering:", before)
print("samples after landmark target filtering:", after)
print("removed samples:", before - after)
print("landmark target count summary:")
print(meta["n_lincs_targets_landmark"].describe())

display(meta[[
    "sample_id",
    "cell_iname",
    "cmap_name_x",
    "chembl_target_gene_list",
    "target_genes_lincs_landmark",
    "target_positions_lincs_landmark",
    "n_lincs_targets_landmark",
]].head())


samples before landmark target filtering: 4800
samples after landmark target filtering: 4800
removed samples: 0
landmark target count summary:
count    4800.000000
mean        1.494792
std         0.862000
min         1.000000
25%         1.000000
50%         1.000000
75%         2.000000
max         5.000000
Name: n_lincs_targets_landmark, dtype: float64


,sample_id,cell_iname,cmap_name_x,chembl_target_gene_list,target_genes_lincs_landmark,target_positions_lincs_landmark,n_lincs_targets_landmark
3,ABY001_A549_XH:BRD-K70511574:0.625:24,A549,HMN-214,[PLK1],[PLK1],[313],1
4,ABY001_A549_XH:BRD-K70511574:10:24,A549,HMN-214,[PLK1],[PLK1],[313],1
5,ABY001_A549_XH:BRD-K70511574:2.5:24,A549,HMN-214,[PLK1],[PLK1],[313],1
6,ABY001_A549_XH:BRD-K81418486:0.625:24,A549,vorinostat,"[HDAC1, HDAC2, HDAC3, HDAC6]","[HDAC2, HDAC6]","[194, 604]",2
7,ABY001_A549_XH:BRD-K81418486:10:24,A549,vorinostat,"[HDAC1, HDAC2, HDAC3, HDAC6]","[HDAC2, HDAC6]","[194, 604]",2


## 6. ctl_vehicle metadata 선택

현재 학습 sample에 등장하는 cell line만 대상으로 `ctl_vehicle` sample을 선택합니다.

In [12]:
ctl_meta_vehicle = ctl_meta[ctl_meta["pert_type"] == "ctl_vehicle"].copy()
ctl_meta_vehicle["sig_id"] = ctl_meta_vehicle["sig_id"].astype(str)
ctl_meta_vehicle["cell_iname"] = ctl_meta_vehicle["cell_iname"].astype(str)
ctl_meta_vehicle["pert_time_num"] = pd.to_numeric(ctl_meta_vehicle["pert_time"], errors="coerce")

needed_cells = sorted(meta["cell_iname"].astype(str).unique().tolist())
ctl_meta_vehicle = ctl_meta_vehicle[ctl_meta_vehicle["cell_iname"].isin(needed_cells)].copy()

print("needed cells:", needed_cells)
print("ctl_vehicle candidates:", ctl_meta_vehicle.shape)
print(ctl_meta_vehicle["cell_iname"].value_counts())
display(ctl_meta_vehicle[["sig_id", "cell_iname", "pert_type", "pert_time_num"]].head())

needed cells: ['A549', 'MCF7', 'PC3']
ctl_vehicle candidates: (11108, 38)
cell_iname
MCF7    4218
PC3     3562
A549    3328
Name: count, dtype: int64


,sig_id,cell_iname,pert_type,pert_time_num
603,HOG001_MCF7_24H:H23,MCF7,ctl_vehicle,24.0
604,HOG001_MCF7_24H:D23,MCF7,ctl_vehicle,24.0
605,HOG001_MCF7_6H:B24,MCF7,ctl_vehicle,6.0
607,DOSBIO002_PC3_24H:A06,PC3,ctl_vehicle,24.0
608,HOG003_A549_24H:O01,A549,ctl_vehicle,24.0


## 7. GCTX에 실제 존재하는 ctl_vehicle sig_id만 남기기

`siginfo_beta.txt`에는 있어도 `CTL_GCTX`에 없는 `sig_id`가 있을 수 있습니다.  
따라서 `parse(..., cid=...)` 전에 반드시 GCTX column metadata와 교집합을 취합니다.

In [13]:
gctx_col_meta_obj = parse(CTL_GCTX, col_meta_only=True)

# cmapPy 버전에 따라 DataFrame 또는 GCToo-like object가 반환될 수 있음
if hasattr(gctx_col_meta_obj, "col_metadata_df"):
    gctx_col_meta = gctx_col_meta_obj.col_metadata_df
else:
    gctx_col_meta = gctx_col_meta_obj

gctx_cids = set(gctx_col_meta.index.astype(str))

candidate_ctl_ids = ctl_meta_vehicle["sig_id"].astype(str).drop_duplicates().tolist()
available_ctl_ids = [sid for sid in candidate_ctl_ids if sid in gctx_cids]
missing_ctl_ids = [sid for sid in candidate_ctl_ids if sid not in gctx_cids]

print("candidate ctl_vehicle ids:", len(candidate_ctl_ids))
print("available in GCTX:", len(available_ctl_ids))
print("missing in GCTX:", len(missing_ctl_ids))
if missing_ctl_ids:
    print("example missing ids:", missing_ctl_ids[:10])

ctl_meta_vehicle = ctl_meta_vehicle[ctl_meta_vehicle["sig_id"].isin(available_ctl_ids)].copy()

if len(available_ctl_ids) == 0:
    raise ValueError("CTL_GCTX에 사용 가능한 ctl_vehicle sig_id가 없습니다. GCTX 파일 또는 siginfo/GCTX 버전을 확인하세요.")

print("filtered ctl_meta_vehicle:", ctl_meta_vehicle.shape)
print(ctl_meta_vehicle["cell_iname"].value_counts())

candidate ctl_vehicle ids: 11108
available in GCTX: 11108
missing in GCTX: 0
filtered ctl_meta_vehicle: (11108, 38)
cell_iname
MCF7    4218
PC3     3562
A549    3328
Name: count, dtype: int64


## 8. ctl_vehicle expression 로딩

`parse()` 결과는 `GCToo` 객체입니다. 실제 matrix는 `ctl_gct.data_df`에 있습니다.  
여기서는 sample × gene 형태로 쓰기 위해 transpose합니다.

In [14]:
ctl_gct = parse(
    CTL_GCTX,
    rid=landmark_gene_ids,
    cid=available_ctl_ids,
)

# 원본: genes × sig_id
# 변환: sig_id × genes
ctl_expr = ctl_gct.data_df.T.copy()
ctl_expr.index = ctl_expr.index.astype(str)
ctl_expr.columns = ctl_expr.columns.astype(str)

print("ctl_expr:", ctl_expr.shape)
print("rows = ctl_vehicle sig_id, columns = landmark genes")
display(ctl_expr.head())

ctl_expr: (11108, 978)
rows = ctl_vehicle sig_id, columns = landmark genes


rid,10007,1001,10013,10038,10046,10049,10051,10057,10058,10059,...,9918,9924,9926,9928,993,994,9943,9961,998,9988
cid,,,,,,,,,,,,,,,,,,,,,
ABY001_A549_XH:DMSO:-666:24,-0.530126,-0.384253,0.360599,0.016843,0.122452,-0.824065,0.247194,0.060323,-0.931731,0.388555,...,0.357877,0.239582,0.143827,0.028286,-0.302712,0.279300,-0.346955,0.391272,0.194382,0.120898
ABY001_A549_XH:DMSO:-666:3,-0.102269,0.502410,-0.706335,-0.513487,-0.442772,0.816279,-0.716717,-0.368180,0.220988,-0.286317,...,-0.375488,-0.606427,-0.561836,0.134261,0.209687,0.010010,-0.382660,-1.158866,-0.383633,-0.745071
ABY001_PC3_XH:DMSO:-666:24,-0.832505,-0.597987,0.335407,0.566547,0.507520,-0.059457,-0.391272,1.318135,-0.503444,0.664435,...,0.023058,0.676950,1.102764,0.132546,0.277114,0.155392,0.404356,0.588467,0.814957,-0.023549
ABY001_PC3_XH:DMSO:-666:3,0.258661,0.075298,-0.256646,-0.215045,-0.459103,-0.323917,0.703211,-0.068921,-0.418130,-0.200083,...,0.680856,0.061385,-0.137827,-0.747464,0.026416,0.419756,0.369563,-0.655815,0.440149,0.350315
AML001_PC3_6H:A17,1.585900,0.170700,1.958950,1.093600,0.583100,0.806650,0.961450,-0.502150,0.981650,-0.223300,...,-0.030550,0.104200,-0.362450,0.922800,0.630200,0.059250,-0.616300,-0.600600,-0.118000,0.921100


## 9. cell line별 ctl_vehicle 평균 profile 생성

동일한 cell line에 여러 `ctl_vehicle` signature가 있으면 gene별 평균을 사용합니다.

In [15]:
# ctl_expr index에 실제 존재하는 sig_id만 재확인
available_ctl_ids_loaded = set(ctl_expr.index.astype(str))
ctl_meta_vehicle = ctl_meta_vehicle[ctl_meta_vehicle["sig_id"].isin(available_ctl_ids_loaded)].copy()

print("matched ctl_meta_vehicle:", ctl_meta_vehicle.shape)
print(ctl_meta_vehicle["cell_iname"].value_counts())

cell_to_xd = {}

for cell, group in ctl_meta_vehicle.groupby("cell_iname"):
    ids = group["sig_id"].astype(str).tolist()
    ids = [sid for sid in ids if sid in ctl_expr.index]
    if len(ids) == 0:
        continue

    # control_sample × gene → gene별 평균
    x_d = ctl_expr.loc[ids].mean(axis=0).to_numpy(dtype=np.float32)
    cell_to_xd[cell] = x_d

print("cell_to_xd keys:", cell_to_xd.keys())

if len(cell_to_xd) == 0:
    raise ValueError("cell_to_xd가 비어 있습니다. ctl_meta_vehicle의 sig_id와 ctl_expr.index 매칭을 확인하세요.")

matched ctl_meta_vehicle: (11108, 38)
cell_iname
MCF7    4218
PC3     3562
A549    3328
Name: count, dtype: int64
cell_to_xd keys: dict_keys(['A549', 'MCF7', 'PC3'])


## 10. ChEMBL target mask `U_prime` 생성

각 sample마다 ChEMBL target gene 위치를 1로 표시한 978차원 binary vector를 만듭니다.

In [16]:
U_prime = np.zeros((meta.shape[0], n_genes), dtype=np.float32)

for i, positions in enumerate(meta["target_positions_lincs_landmark"]):
    for pos in positions:
        if 0 <= pos < n_genes:
            U_prime[i, pos] = 1.0

U_prime_df = pd.DataFrame(
    U_prime,
    index=meta["sample_id"].astype(str),
    columns=[f"u_prime_{gid}" for gid in landmark_gene_ids]
)

print("U_prime_df:", U_prime_df.shape)
print("nonzero per sample:")
print(U_prime_df.sum(axis=1).describe())
print("Expected number of genes:", n_genes)
display(U_prime_df.head())


U_prime_df: (4800, 978)
nonzero per sample:
count    4800.000000
mean        1.494792
std         0.862000
min         1.000000
25%         1.000000
50%         1.000000
75%         2.000000
max         5.000000
dtype: float64
Expected number of genes: 978


,u_prime_16,u_prime_23,u_prime_25,u_prime_30,u_prime_39,u_prime_47,u_prime_102,u_prime_128,u_prime_142,u_prime_154,...,u_prime_94239,u_prime_116832,u_prime_124583,u_prime_147179,u_prime_148022,u_prime_200081,u_prime_200734,u_prime_256364,u_prime_375346,u_prime_388650
sample_id,,,,,,,,,,,,,,,,,,,,,
ABY001_A549_XH:BRD-K70511574:0.625:24,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ABY001_A549_XH:BRD-K70511574:10:24,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ABY001_A549_XH:BRD-K70511574:2.5:24,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ABY001_A549_XH:BRD-K81418486:0.625:24,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ABY001_A549_XH:BRD-K81418486:10:24,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 11. sample별 ctl_vehicle baseline expression matrix 생성

각 sample의 `cell_iname`에 해당하는 cell line 평균 ctl_vehicle profile을 붙입니다.  
ctl_vehicle profile이 없는 cell line의 sample은 제거합니다.

In [17]:
meta["cell_iname"] = meta["cell_iname"].astype(str)
meta = meta[meta["cell_iname"].isin(cell_to_xd.keys())].copy()

if meta.empty:
    raise ValueError("ctl_vehicle profile이 있는 cell line과 meta가 겹치지 않습니다.")

X_disease_expr = np.vstack([
    cell_to_xd[cell] for cell in meta["cell_iname"]
]).astype(np.float32)

X_disease_expr_df = pd.DataFrame(
    X_disease_expr,
    index=meta["sample_id"].astype(str),
    columns=[f"xd_{gid}" for gid in landmark_gene_ids]
)

# U_prime도 meta 필터링 후 index에 맞춤
U_prime_df = U_prime_df.loc[meta["sample_id"].astype(str)].copy()

print("meta after ctl profile filtering:", meta.shape)
print("X_disease_expr_df:", X_disease_expr_df.shape)
print("U_prime_df:", U_prime_df.shape)
print("Index aligned:", (X_disease_expr_df.index == U_prime_df.index).all())
display(X_disease_expr_df.head())

meta after ctl profile filtering: (4800, 50)
X_disease_expr_df: (4800, 978)
U_prime_df: (4800, 978)
Index aligned: True


,xd_16,xd_23,xd_25,xd_30,xd_39,xd_47,xd_102,xd_128,xd_142,xd_154,...,xd_94239,xd_116832,xd_124583,xd_147179,xd_148022,xd_200081,xd_200734,xd_256364,xd_375346,xd_388650
sample_id,,,,,,,,,,,,,,,,,,,,,
ABY001_A549_XH:BRD-K70511574:0.625:24,0.023052,-0.093788,0.235705,0.115597,0.012831,0.167165,-0.023823,0.028773,0.024012,0.051478,...,-0.04335,0.037018,0.031926,0.017378,-0.05121,0.009414,-0.073997,-0.015591,-0.00608,0.001432
ABY001_A549_XH:BRD-K70511574:10:24,0.023052,-0.093788,0.235705,0.115597,0.012831,0.167165,-0.023823,0.028773,0.024012,0.051478,...,-0.04335,0.037018,0.031926,0.017378,-0.05121,0.009414,-0.073997,-0.015591,-0.00608,0.001432
ABY001_A549_XH:BRD-K70511574:2.5:24,0.023052,-0.093788,0.235705,0.115597,0.012831,0.167165,-0.023823,0.028773,0.024012,0.051478,...,-0.04335,0.037018,0.031926,0.017378,-0.05121,0.009414,-0.073997,-0.015591,-0.00608,0.001432
ABY001_A549_XH:BRD-K81418486:0.625:24,0.023052,-0.093788,0.235705,0.115597,0.012831,0.167165,-0.023823,0.028773,0.024012,0.051478,...,-0.04335,0.037018,0.031926,0.017378,-0.05121,0.009414,-0.073997,-0.015591,-0.00608,0.001432
ABY001_A549_XH:BRD-K81418486:10:24,0.023052,-0.093788,0.235705,0.115597,0.012831,0.167165,-0.023823,0.028773,0.024012,0.051478,...,-0.04335,0.037018,0.031926,0.017378,-0.05121,0.009414,-0.073997,-0.015591,-0.00608,0.001432


## 12. 최종 입력 테이블 구성 및 저장

최종 입력은 다음 정보를 포함합니다.

- `xd_*`: cell line별 평균 `ctl_vehicle` baseline/diseased-like expression, 978 landmark genes
- `u_prime_*`: ChEMBL target gene mask, 978 landmark genes
- `pert_id`, `cell_iname`, `pert_dose_num`, `pert_time_num`: treatment condition metadata

따라서 gene 관련 feature는 모두 LINCS 978 landmark gene 기준으로 정렬됩니다.


In [18]:
treatment_cols = [c for c in ["pert_id", "cell_iname", "pert_dose_num", "pert_time_num"] if c in meta.columns]
X_treatment_meta = meta.set_index("sample_id")[treatment_cols].copy()
X_treatment_meta.index = X_treatment_meta.index.astype(str)

# Safety checks: both matrices must use the same 978 landmark genes.
assert X_disease_expr_df.shape[1] == n_genes, "ctl_vehicle expression gene dimension is not 978 landmark genes."
assert U_prime_df.shape[1] == n_genes, "U_prime gene dimension is not 978 landmark genes."
assert list(X_disease_expr_df.columns) == [f"xd_{gid}" for gid in landmark_gene_ids]
assert list(U_prime_df.columns) == [f"u_prime_{gid}" for gid in landmark_gene_ids]

X_merged = pd.concat(
    [
        X_disease_expr_df,
        U_prime_df,
        X_treatment_meta.loc[X_disease_expr_df.index]
    ],
    axis=1
)

metadata_merged = meta.set_index("sample_id").loc[X_merged.index].copy()

print("X_merged:", X_merged.shape)
print("metadata_merged:", metadata_merged.shape)
print("Index aligned:", (X_merged.index == metadata_merged.index).all())
print("xd feature count:", len([c for c in X_merged.columns if c.startswith("xd_")]))
print("u_prime feature count:", len([c for c in X_merged.columns if c.startswith("u_prime_")]))
display(X_merged.head())


X_merged: (4800, 1960)
metadata_merged: (4800, 49)
Index aligned: True
xd feature count: 978
u_prime feature count: 978


,xd_16,xd_23,xd_25,xd_30,xd_39,xd_47,xd_102,xd_128,xd_142,xd_154,...,u_prime_148022,u_prime_200081,u_prime_200734,u_prime_256364,u_prime_375346,u_prime_388650,pert_id,cell_iname,pert_dose_num,pert_time_num
sample_id,,,,,,,,,,,,,,,,,,,,,
ABY001_A549_XH:BRD-K70511574:0.625:24,0.023052,-0.093788,0.235705,0.115597,0.012831,0.167165,-0.023823,0.028773,0.024012,0.051478,...,0.0,0.0,0.0,0.0,0.0,0.0,BRD-K70511574,A549,0.625,24.0
ABY001_A549_XH:BRD-K70511574:10:24,0.023052,-0.093788,0.235705,0.115597,0.012831,0.167165,-0.023823,0.028773,0.024012,0.051478,...,0.0,0.0,0.0,0.0,0.0,0.0,BRD-K70511574,A549,10.000,24.0
ABY001_A549_XH:BRD-K70511574:2.5:24,0.023052,-0.093788,0.235705,0.115597,0.012831,0.167165,-0.023823,0.028773,0.024012,0.051478,...,0.0,0.0,0.0,0.0,0.0,0.0,BRD-K70511574,A549,2.500,24.0
ABY001_A549_XH:BRD-K81418486:0.625:24,0.023052,-0.093788,0.235705,0.115597,0.012831,0.167165,-0.023823,0.028773,0.024012,0.051478,...,0.0,0.0,0.0,0.0,0.0,0.0,BRD-K81418486,A549,0.625,24.0
ABY001_A549_XH:BRD-K81418486:10:24,0.023052,-0.093788,0.235705,0.115597,0.012831,0.167165,-0.023823,0.028773,0.024012,0.051478,...,0.0,0.0,0.0,0.0,0.0,0.0,BRD-K81418486,A549,10.000,24.0


| 파일명                                       | 내용                                                   |
| ----------------------------------------- | ---------------------------------------------------- |
| `metadata_merged_chembl_ctl_landmark.csv` | 최종 학습 sample metadata                                |
| `X_merged_chembl_ctl_landmark.pkl`        | 최종 입력 feature 전체                                     |
| `X_disease_expr_ctl_vehicle_landmark.pkl` | `ctl_vehicle` 기반 disease/baseline expression feature |
| `U_prime_chembl_targets_landmark.pkl`     | ChEMBL target gene mask feature                      |
| `merged_chembl_ctl_landmark_arrays.npz`   | 모델 학습용 numpy array 묶음                                |


In [19]:
metadata_merged.to_csv(os.path.join(OUT_DIR, "metadata_merged_chembl_ctl_landmark.csv"), index=True)
X_merged.to_pickle(os.path.join(OUT_DIR, "X_merged_chembl_ctl_landmark.pkl"))
X_disease_expr_df.to_pickle(os.path.join(OUT_DIR, "X_disease_expr_ctl_vehicle_landmark.pkl"))
U_prime_df.to_pickle(os.path.join(OUT_DIR, "U_prime_chembl_targets_landmark.pkl"))

np.savez_compressed(
    os.path.join(OUT_DIR, "merged_chembl_ctl_landmark_arrays.npz"),
    X_disease_expr=X_disease_expr_df.to_numpy(dtype=np.float32),
    U_prime=U_prime_df.to_numpy(dtype=np.float32),
    sample_ids=X_merged.index.astype(str).to_numpy(),
    gene_ids=np.array(landmark_gene_ids, dtype=str),
    gene_symbols=np.array(landmark_gene_symbols, dtype=str),
)

print("Saved files to:", OUT_DIR)
print("- metadata_merged_chembl_ctl_landmark.csv")
print("- X_merged_chembl_ctl_landmark.pkl")
print("- X_disease_expr_ctl_vehicle_landmark.pkl")
print("- U_prime_chembl_targets_landmark.pkl")
print("- merged_chembl_ctl_landmark_arrays.npz")


Saved files to: ./processed/input
- metadata_merged_chembl_ctl_landmark.csv
- X_merged_chembl_ctl_landmark.pkl
- X_disease_expr_ctl_vehicle_landmark.pkl
- U_prime_chembl_targets_landmark.pkl
- merged_chembl_ctl_landmark_arrays.npz


In [ ]:
import os
import numpy as np
import pandas as pd

OUT_DIR = "./processed/result"

metadata_merged = pd.read_csv(os.path.join(OUT_DIR, "metadata_merged_chembl_ctl_landmark.csv"), index_col=0)
X_merged = pd.read_pickle(os.path.join(OUT_DIR, "X_merged_chembl_ctl_landmark.pkl"))
X_disease_expr_df = pd.read_pickle(os.path.join(OUT_DIR, "diseased_expr.pkl"))
U_prime_df = pd.read_pickle(os.path.join(OUT_DIR, "target.pkl"))

print("metadata_merged")
display(metadata_merged.head())

print("X_merged")
display(X_merged.head())

print("X_disease_expr_df")
display(X_disease_expr_df.head())

print("U_prime_df")
display(U_prime_df.head())

metadata_merged


,bead_batch,nearest_dose,pert_dose,pert_dose_unit,pert_idose,pert_itime,pert_time,pert_time_unit,cell_mfc_name,pert_mfc_id,...,molecule_chembl_id,chembl_target_genes,n_chembl_targets,target_genes_lincs,target_positions_lincs,n_lincs_targets,chembl_target_gene_list,target_genes_lincs_landmark,target_positions_lincs_landmark,n_lincs_targets_landmark
sample_id,,,,,,,,,,,,,,,,,,,,,
ABY001_A549_XH:BRD-K70511574:0.625:24,b15,0.66,0.625,uM,0.66 uM,24 h,24.0,h,A549,BRD-K70511574,...,CHEMBL3991927,PLK1,1,PLK1,2467,1,['PLK1'],['PLK1'],[313],1
ABY001_A549_XH:BRD-K70511574:10:24,b15,10.00,10.000,uM,10 uM,24 h,24.0,h,A549,BRD-K70511574,...,CHEMBL3991927,PLK1,1,PLK1,2467,1,['PLK1'],['PLK1'],[313],1
ABY001_A549_XH:BRD-K70511574:2.5:24,b15,2.50,2.500,uM,2.5 uM,24 h,24.0,h,A549,BRD-K70511574,...,CHEMBL3991927,PLK1,1,PLK1,2467,1,['PLK1'],['PLK1'],[313],1
ABY001_A549_XH:BRD-K81418486:0.625:24,b15,0.66,0.625,uM,0.66 uM,24 h,24.0,h,A549,BRD-K81418486,...,CHEMBL98,"HDAC1,HDAC2,HDAC3,HDAC6",4,"HDAC1,HDAC2,HDAC3,HDAC6","4562,2348,7096,2758",4,"['HDAC1', 'HDAC2', 'HDAC3', 'HDAC6']","['HDAC2', 'HDAC6']","[194, 604]",2
ABY001_A549_XH:BRD-K81418486:10:24,b15,10.00,10.000,uM,10 uM,24 h,24.0,h,A549,BRD-K81418486,...,CHEMBL98,"HDAC1,HDAC2,HDAC3,HDAC6",4,"HDAC1,HDAC2,HDAC3,HDAC6","4562,2348,7096,2758",4,"['HDAC1', 'HDAC2', 'HDAC3', 'HDAC6']","['HDAC2', 'HDAC6']","[194, 604]",2


X_merged


,xd_16,xd_23,xd_25,xd_30,xd_39,xd_47,xd_102,xd_128,xd_142,xd_154,...,u_prime_148022,u_prime_200081,u_prime_200734,u_prime_256364,u_prime_375346,u_prime_388650,pert_id,cell_iname,pert_dose_num,pert_time_num
sample_id,,,,,,,,,,,,,,,,,,,,,
ABY001_A549_XH:BRD-K70511574:0.625:24,0.023052,-0.093788,0.235705,0.115597,0.012831,0.167165,-0.023823,0.028773,0.024012,0.051478,...,0.0,0.0,0.0,0.0,0.0,0.0,BRD-K70511574,A549,0.625,24.0
ABY001_A549_XH:BRD-K70511574:10:24,0.023052,-0.093788,0.235705,0.115597,0.012831,0.167165,-0.023823,0.028773,0.024012,0.051478,...,0.0,0.0,0.0,0.0,0.0,0.0,BRD-K70511574,A549,10.000,24.0
ABY001_A549_XH:BRD-K70511574:2.5:24,0.023052,-0.093788,0.235705,0.115597,0.012831,0.167165,-0.023823,0.028773,0.024012,0.051478,...,0.0,0.0,0.0,0.0,0.0,0.0,BRD-K70511574,A549,2.500,24.0
ABY001_A549_XH:BRD-K81418486:0.625:24,0.023052,-0.093788,0.235705,0.115597,0.012831,0.167165,-0.023823,0.028773,0.024012,0.051478,...,0.0,0.0,0.0,0.0,0.0,0.0,BRD-K81418486,A549,0.625,24.0
ABY001_A549_XH:BRD-K81418486:10:24,0.023052,-0.093788,0.235705,0.115597,0.012831,0.167165,-0.023823,0.028773,0.024012,0.051478,...,0.0,0.0,0.0,0.0,0.0,0.0,BRD-K81418486,A549,10.000,24.0


X_disease_expr_df


,xd_16,xd_23,xd_25,xd_30,xd_39,xd_47,xd_102,xd_128,xd_142,xd_154,...,xd_94239,xd_116832,xd_124583,xd_147179,xd_148022,xd_200081,xd_200734,xd_256364,xd_375346,xd_388650
sample_id,,,,,,,,,,,,,,,,,,,,,
ABY001_A549_XH:BRD-K70511574:0.625:24,0.023052,-0.093788,0.235705,0.115597,0.012831,0.167165,-0.023823,0.028773,0.024012,0.051478,...,-0.04335,0.037018,0.031926,0.017378,-0.05121,0.009414,-0.073997,-0.015591,-0.00608,0.001432
ABY001_A549_XH:BRD-K70511574:10:24,0.023052,-0.093788,0.235705,0.115597,0.012831,0.167165,-0.023823,0.028773,0.024012,0.051478,...,-0.04335,0.037018,0.031926,0.017378,-0.05121,0.009414,-0.073997,-0.015591,-0.00608,0.001432
ABY001_A549_XH:BRD-K70511574:2.5:24,0.023052,-0.093788,0.235705,0.115597,0.012831,0.167165,-0.023823,0.028773,0.024012,0.051478,...,-0.04335,0.037018,0.031926,0.017378,-0.05121,0.009414,-0.073997,-0.015591,-0.00608,0.001432
ABY001_A549_XH:BRD-K81418486:0.625:24,0.023052,-0.093788,0.235705,0.115597,0.012831,0.167165,-0.023823,0.028773,0.024012,0.051478,...,-0.04335,0.037018,0.031926,0.017378,-0.05121,0.009414,-0.073997,-0.015591,-0.00608,0.001432
ABY001_A549_XH:BRD-K81418486:10:24,0.023052,-0.093788,0.235705,0.115597,0.012831,0.167165,-0.023823,0.028773,0.024012,0.051478,...,-0.04335,0.037018,0.031926,0.017378,-0.05121,0.009414,-0.073997,-0.015591,-0.00608,0.001432


U_prime_df


,u_prime_16,u_prime_23,u_prime_25,u_prime_30,u_prime_39,u_prime_47,u_prime_102,u_prime_128,u_prime_142,u_prime_154,...,u_prime_94239,u_prime_116832,u_prime_124583,u_prime_147179,u_prime_148022,u_prime_200081,u_prime_200734,u_prime_256364,u_prime_375346,u_prime_388650
sample_id,,,,,,,,,,,,,,,,,,,,,
ABY001_A549_XH:BRD-K70511574:0.625:24,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ABY001_A549_XH:BRD-K70511574:10:24,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ABY001_A549_XH:BRD-K70511574:2.5:24,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ABY001_A549_XH:BRD-K81418486:0.625:24,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ABY001_A549_XH:BRD-K81418486:10:24,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [7]:
target_ids = metadata_merged.index.astype(str)

X_lincs = pd.read_csv("./lincs/X.csv", index_col = 0)
y_lincs = pd.read_parquet("./lincs/y.parquet")

X_lincs.index = X_lincs.index.astype(str)
y_lincs.index = y_lincs.index.astype(str)

common_ids = target_ids.intersection(X_lincs.index).intersection(y_lincs.index)

print("common sample count:", len(common_ids))

X_lincs_matched = X_lincs.loc[common_ids].copy()
y_lincs_matched = y_lincs.loc[common_ids].copy()

X_lincs_matched = X_lincs_matched.loc[common_ids]
y_lincs_matched = y_lincs_matched.loc[common_ids]

print("Matched X_lincs:", X_lincs_matched.shape)
print("Matched y_lincs:", y_lincs_matched.shape)
print("Index aligned:", X_lincs_matched.index.equals(y_lincs_matched.index))

X_lincs_matched.to_csv(
    "./processed/processed_X.csv",
    index=True
)

y_lincs_matched.to_pickle(
    "./processed/processed_y.pkl"
)


common sample count: 4800
Matched X_lincs: (4800, 4)
Matched y_lincs: (4800, 978)
Index aligned: True
